# Phase 1 — Data Collection & Merging

Predicting financial distress in U.S. private nonprofit 4-year colleges.

This notebook builds the Phase 1 panel:

1. Download FSA Financial Responsibility Composite Scores (every configured year)
2. Download IPEDS complete data files (HD, EF, F2), filtered to private nonprofit 4-year institutions
3. Pull Form 990 financials by EIN from the ProPublica Nonprofit Explorer API (cached, rate-limited)
4. Merge everything into one institution-year panel and label `distressed_next_year`
5. Print a data quality report
6. Spot-check Oakwood University (EIN 63-0366652) against its own Form 990s

See `../NOTES.md` for a running log of problems hit while building this and how they were fixed,
and `../README.md` for the project layout and known limitations.

In [1]:
import sys
sys.path.insert(0, "../scripts")

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

from config import YEARS, OAKWOOD_EIN
print(f"Phase 1 year range: {YEARS[0]}-{YEARS[-1]} ({len(YEARS)} years)")

Phase 1 year range: 2014-2023 (10 years)


## Step 1 — FSA Financial Responsibility Composite Scores

Downloads one `.xls` file per award year from studentaid.gov and parses it into a single table of
`opeid`, `institution_name`, `fiscal_year`, `composite_score`. See NOTES.md for why the column layout
had to be matched by normalized name rather than position (it changes almost every year).

In [2]:
import fetch_composite_scores as fcs

fcs.download_all()
composite_df = fcs.parse_all()
print(composite_df.shape)
composite_df.head()

(33522, 7)


,opeid,institution_name,institution_type,institution_fiscal_year_end,composite_score,score_year,fiscal_year
0,00884300,Alaska Bible College,Non Profit,2014-05-31,1.4,2014,2014
1,02541000,Alaska Career College,Proprietary,2013-12-31,2.4,2014,2013
2,04138600,Alaska Christian College,Non Profit,2014-05-31,2.6,2014,2014
3,00106100,Alaska Pacific University,Non Profit,2014-06-30,2.2,2014,2014
4,02576900,Charter College,Proprietary,2013-12-31,1.3,2014,2013


In [3]:
composite_df["composite_score"].describe()

count    33522.000000
mean         2.264283
std          0.826400
min         -1.000000
25%          1.800000
50%          2.500000
75%          3.000000
max          3.000000
Name: composite_score, dtype: float64

## Step 2 — IPEDS complete data files (HD, EF, F2)

For each year: downloads the institutional characteristics (HD), fall enrollment (EF), and private
nonprofit/FASB finance (F2) complete data files, filters HD to `SECTOR == 2`
("Private not-for-profit, 4-year or above"), and merges enrollment + finance fields onto that
institution universe for the year.

In [4]:
import fetch_ipeds as fip

ipeds_df = fip.build_panel()
print(ipeds_df.shape)
ipeds_df.head()

(16813, 17)


,unitid,institution_name_ipeds,opeid,ein,state,SECTOR,CONTROL,ICLEVEL,hbcu,total_enrollment,total_assets,total_liabilities,total_net_assets,total_revenues,total_expenses,net_assets_end_of_year,ipeds_year
0,100690,Amridge University,02503400,237034324,AL,2,2,1,2,625.0,13848241.0,3019785.0,10828456.0,10236876.0,8348077.0,10828456.0,2014
1,100937,Birmingham Southern College,00101200,630708730,AL,2,2,1,2,1185.0,195685360.0,72417000.0,123268360.0,48876673.0,45134655.0,123268360.0,2014
2,101073,Concordia College Alabama,01055400,630389941,AL,2,2,1,1,546.0,17760303.0,3817334.0,13942969.0,12860451.0,13032162.0,13942969.0,2014
3,101189,Faulkner University,00100300,630329409,AL,2,2,1,2,3335.0,89110757.0,42162603.0,46948154.0,42846547.0,44565242.0,46948154.0,2014
4,101435,Huntingdon College,00101900,630288841,AL,2,2,1,2,1160.0,81389270.0,27527679.0,53861591.0,22582643.0,20052724.0,53861591.0,2014


In [5]:
ipeds_df.groupby("ipeds_year").size().rename("n_institutions")

ipeds_year
2014    1695
2015    1717
2016    1705
2017    1701
2018    1687
2019    1685
2020    1673
2021    1655
2022    1656
2023    1639
Name: n_institutions, dtype: int64

## Step 3 & 4 — Form 990 financials (ProPublica) + merge into one panel

`build_full_panel` attaches the composite score (same `opeid` + fiscal year), the next year's
composite score (used for the label below), and each institution's Form 990 financials for that
fiscal year looked up by EIN via the ProPublica Nonprofit Explorer API.

The Form 990 fetch is rate-limited and caches every response to
`../data/raw/form990_cache/{ein}.json`, so only the first run pays the full crawl cost — re-running
this cell after the first time is effectively instant.

`distressed_next_year` is 1 if the institution's **composite score in fiscal year t+1** is below 1.5,
0 if it is posted and >= 1.5, and missing if no score was posted for that institution in year t+1
(e.g. it closed, merged, or simply has not been disclosed yet).

In [6]:
from build_panel import build_full_panel

panel = build_full_panel(ipeds_df, composite_df)
print(panel.shape)
panel.head()

(16813, 26)


,unitid,institution_name_ipeds,opeid,ein,state,SECTOR,CONTROL,ICLEVEL,hbcu,total_enrollment,total_assets,total_liabilities,total_net_assets,total_revenues,total_expenses,net_assets_end_of_year,fiscal_year,composite_score,institution_name_fsa,next_year_score,distressed_next_year,f990_total_revenue,f990_total_expenses,f990_total_assets_end,f990_total_liabilities_end,f990_net_assets_end
0,100690,Amridge University,02503400,237034324,AL,2,2,1,2,625.0,13848241.0,3019785.0,10828456.0,10236876.0,8348077.0,10828456.0,2014,3.0,Amridge University,3.0,0.0,10631300.0,8742501.0,13848241.0,3019785.0,10828456.0
1,100937,Birmingham Southern College,00101200,630708730,AL,2,2,1,2,1185.0,195685360.0,72417000.0,123268360.0,48876673.0,45134655.0,123268360.0,2014,2.9,Birmingham-Southern College,2.0,0.0,NaN,NaN,NaN,NaN,NaN
2,101073,Concordia College Alabama,01055400,630389941,AL,2,2,1,1,546.0,17760303.0,3817334.0,13942969.0,12860451.0,13032162.0,13942969.0,2014,2.6,Concordia College Alabama,NaN,NaN,13977422.0,14290697.0,17760303.0,3817334.0,13942969.0
3,101189,Faulkner University,00100300,630329409,AL,2,2,1,2,3335.0,89110757.0,42162603.0,46948154.0,42846547.0,44565242.0,46948154.0,2014,2.2,Faulkner University,2.7,0.0,52612603.0,54918760.0,88544132.0,41927644.0,46616488.0
4,101435,Huntingdon College,00101900,630288841,AL,2,2,1,2,1160.0,81389270.0,27527679.0,53861591.0,22582643.0,20052724.0,53861591.0,2014,2.0,Huntingdon College,2.9,0.0,29957155.0,30039542.0,81389270.0,27527679.0,53861591.0


In [7]:
panel.to_csv("../data/processed/panel.csv", index=False)
print("Saved ../data/processed/panel.csv")

Saved ../data/processed/panel.csv


## Step 5 — Data quality report

In [8]:
from quality_report import print_report

print_report(panel, ipeds_df, composite_df)

DATA QUALITY REPORT
Panel rows (institution-year observations): 16813
Distinct institutions (UNITID):              1901
Distinct fiscal years:                        10
Fiscal year range:                            2014-2023

--- Match rates between sources ---
IPEDS private-nonprofit-4yr rows:                     16813
  ... matched to a composite score (same opeid+year): 14607 (86.9% of panel rows)
Distinct OPEIDs in composite-score files:             4284
Distinct OPEIDs in IPEDS private-nonprofit-4yr universe: 1874
  ... overlap:                                        1658

Rows with a non-null EIN:                             16813
  ... matched to a Form 990 filing for that fiscal year: 14050 (83.6% of rows with an EIN)

--- Label balance (distressed_next_year) ---
Rows with a usable label (score observed in year t+1): 13027 (77.5% of panel)
Share distressed (score < 1.5 next year) among labeled rows: 6.7%

--- Missing values per column ---
                            missing_cou

                              count          mean           std          min          25%          50%           75%           max
unitid                      16813.0  2.264767e+05  1.090667e+05     100690.0    155414.00     195164.0  2.314200e+05  4.996350e+05
SECTOR                      16813.0  2.000000e+00  0.000000e+00          2.0         2.00          2.0  2.000000e+00  2.000000e+00
CONTROL                     16813.0  2.000000e+00  0.000000e+00          2.0         2.00          2.0  2.000000e+00  2.000000e+00
ICLEVEL                     16813.0  1.000000e+00  0.000000e+00          1.0         1.00          1.0  1.000000e+00  1.000000e+00
hbcu                        16813.0  1.970618e+00  1.688799e-01          1.0         2.00          2.0  2.000000e+00  2.000000e+00
total_enrollment            16480.0  2.557168e+03  6.354897e+03          1.0       308.00       1114.0  2.522000e+03  1.850150e+05
total_assets                15393.0  5.869377e+08  3.177914e+09          0.0  20881

## Step 6 — Oakwood University (EIN 63-0366652)

Oakwood University's rows in the merged panel, for cross-checking against its own Form 990 filings.

In [9]:
oakwood = panel[panel["ein"] == OAKWOOD_EIN].sort_values("fiscal_year")
oakwood

,unitid,institution_name_ipeds,opeid,ein,state,SECTOR,CONTROL,ICLEVEL,hbcu,total_enrollment,total_assets,total_liabilities,total_net_assets,total_revenues,total_expenses,net_assets_end_of_year,fiscal_year,composite_score,institution_name_fsa,next_year_score,distressed_next_year,f990_total_revenue,f990_total_expenses,f990_total_assets_end,f990_total_liabilities_end,f990_net_assets_end
9,101912,Oakwood University,00103300,630366652,AL,2,2,1,1,1939.0,90161674.0,20654626.0,69507048.0,55327159.0,51670636.0,69507048.0,2014,3.0,Oakwood University,3.0,0.0,60568302.0,58243489.0,90161674.0,20654626.0,69507048.0
1704,101912,Oakwood University,00103300,630366652,AL,2,2,1,1,1749.0,93725737.0,18889069.0,74836668.0,57549593.0,52219973.0,74836668.0,2015,3.0,Oakwood University,2.8,0.0,64335232.0,59059088.0,93725737.0,18889069.0,74836668.0
3421,101912,Oakwood University,00103300,630366652,AL,2,2,1,1,1794.0,93924759.0,17749201.0,76175558.0,53579750.0,52240860.0,76175558.0,2016,2.8,Oakwood University,3.0,0.0,60732332.0,58396037.0,93924759.0,17749201.0,76175558.0
5127,101912,Oakwood University,00103300,630366652,AL,2,2,1,1,1711.0,97726924.0,17085268.0,80641656.0,55721197.0,51255099.0,80641656.0,2017,3.0,Oakwood University,2.7,0.0,60605674.0,57971822.0,97726924.0,17085268.0,80641656.0
6828,101912,Oakwood University,00103300,630366652,AL,2,2,1,1,1636.0,99736770.0,16516728.0,83220042.0,57364668.0,54786282.0,83220042.0,2018,2.7,Oakwood University,NaN,NaN,62425532.0,60681506.0,99736770.0,16516728.0,83220042.0
8514,101912,Oakwood University,00103300,630366652,AL,2,2,1,1,1526.0,101474881.0,16741379.0,84733502.0,57079560.0,55566100.0,84733502.0,2019,NaN,NaN,2.5,0.0,63581649.0,62216416.0,101474881.0,16741379.0,84733502.0
10199,101912,Oakwood University,00103300,630366652,AL,2,2,1,1,1374.0,104408271.0,19154713.0,85253558.0,58363294.0,57843238.0,85253558.0,2020,2.5,Oakwood University,3.0,0.0,65439596.0,64752168.0,104408271.0,19154713.0,85253558.0
11872,101912,Oakwood University,00103300,630366652,AL,2,2,1,1,1452.0,115223075.0,21667855.0,93555220.0,61654689.0,53353027.0,93555220.0,2021,3.0,Oakwood University,2.2,0.0,64178393.0,61467355.0,115223075.0,21667855.0,93555220.0
13526,101912,Oakwood University,00103300,630366652,AL,2,2,1,1,1470.0,128063334.0,43426280.0,84637054.0,54456136.0,63374302.0,84637054.0,2022,2.2,Oakwood University,NaN,NaN,68745212.0,72386999.0,120064681.0,35427627.0,84637054.0
15182,101912,Oakwood University,00103300,630366652,AL,2,2,1,1,1319.0,118855484.0,41884390.0,76971094.0,58953807.0,66619767.0,76971094.0,2023,NaN,NaN,NaN,NaN,67340745.0,77013728.0,111446919.0,34298132.0,77148787.0


In [10]:
# Cross-check against the raw ProPublica record (all filings, not just the ones matched into the panel)
from fetch_form990 import get_organization

oakwood_990 = get_organization(OAKWOOD_EIN)
oakwood_filings = pd.DataFrame(oakwood_990["filings_with_data"])[
    ["tax_prd_yr", "totrevenue", "totfuncexpns", "totassetsend", "totliabend", "totnetassetend"]
].sort_values("tax_prd_yr")
oakwood_filings

,tax_prd_yr,totrevenue,totfuncexpns,totassetsend,totliabend,totnetassetend
12,2011,56320518,53441952,71057268,14214281,56842987
11,2012,58307057,54895323,75005830,14784140,60221690
10,2013,58689847,54014572,79743761,13893236,65850525
9,2014,60568302,58243489,90161674,20654626,69507048
8,2015,64335232,59059088,93725737,18889069,74836668
7,2016,60732332,58396037,93924759,17749201,76175558
6,2017,60605674,57971822,97726924,17085268,80641656
5,2018,62425532,60681506,99736770,16516728,83220042
4,2019,63581649,62216416,101474881,16741379,84733502
3,2020,65439596,64752168,104408271,19154713,85253558
